# 입찰메이트 RAG — 서빙(v2) E2E 평가 (리트리벌 0점 수정본)

**KURE 임베딩 + Phi-4-mini FT 단독, 통합본 단일 청크, 서빙 코드 그대로**

- 청크: `chunks_all.json` / chroma `bidmate_chunks_all` (10,068) / bm25 `bm25_index_bidmate_chunks_all_A-2.pkl`
- 모듈: 서빙 `retrieval.py` + `generation.py` (code 폴더에서 파일 경로로 직접 로드)
- 평가셋: `eval_retrieval_579.csv`
- 지표: Retrieval(Hit@5/MRR/nDCG) + Generation(6지표 Judge) + Release Gate
- 저장: `outputs/<tag>/`

**실행 순서: 위에서부터 `[0]`→`[10]` 순서대로. 세션 리셋 시 `[0]`부터 다시.**

---

### 이전 실행에서 리트리벌이 0점 나온 원인 (수정 완료)
1. **chroma count=0인 빈 컬렉션을 재사용** — `_ensure_chroma()`의 재사용 조건이 `count >= 0`이라 빈 컬렉션(0)도 통과해 tar 재해제를 건너뜀. → **`count >= EXPECT_MIN`일 때만 재사용**하도록 수정.
2. **pre-check가 0을 통과** — `EXPECT_N`이 0이라 `n_chroma == EXPECT_N`이 `0 == 0`으로 통과. → **`n_chroma > 0` assert를 먼저** 추가.
3. **id가 키로 부적합** — eval CSV가 행 579 / 고유 id 499 (같은 id에 다른 question). → **question 기준 dedup**으로 통일.
4. **검색결과명↔정답문서 매칭 진단 셀 추가** — chroma 정상화 후에도 점수가 낮으면 포맷 불일치 확인용.


In [1]:
# [setup-clean] 이전 세션에서 풀린 빈/오염 chroma 로컬 폴더 정리 (선택 — 0점 재발 방지 1순위)
#   직전 실행에서 chroma count=0 으로 나왔다면 반드시 실행한 뒤 [1] 로 진행.
import shutil, os
for p in ['/content/bidmate_chunks_all', '/content/bidmate_retrieval_v1',
          '/content/_chroma_probe', '/content/chroma_db']:
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True)
        print('삭제:', p)
print('정리 완료 — 이제 [0] 설치 → [1] 마운트 순으로 실행')

정리 완료 — 이제 [0] 설치 → [1] 마운트 순으로 실행


In [2]:
# [0] 설치
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [1]부터")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━

In [3]:
# [0b] 진단 — chroma_db(tar 및 로컬) 안의 컬렉션 이름·개수 확인 (마운트 후 1회)
#   여기서 나온 "count > 0" 인 이름을 [1] config 의 COLLECTION_NAME 에 그대로 넣으세요.
import os, shutil, tarfile, gc, chromadb
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass
DRIVE = '/content/drive/MyDrive/data/bidmate'

def list_collections(chroma_dir):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    client = chromadb.PersistentClient(path=chroma_dir)
    cols = client.list_collections()
    print(f'  경로: {chroma_dir}')
    if not cols: print('  (컬렉션 없음)')
    for c in cols:
        try: print(f'  - {c.name:32} count={client.get_collection(c.name).count():,}')
        except Exception as e: print(f'  - {c.name:32} (count 실패: {e})')

# 1) 로컬에 이미 풀려있으면 표시 (★ count 0 이면 그 폴더는 쓰면 안 됨)
for p in ['/content/bidmate_chunks_all/chroma_db', '/content/chroma_db']:
    if os.path.isdir(p):
        print('▶ 로컬 chroma'); list_collections(p)

# 2) 드라이브 tar.gz 를 임시로 풀어 확인 (← 정상 데이터의 출처)
tar = f'{DRIVE}/chroma_db.tar.gz'
tmp = '/content/_chroma_probe'
if os.path.exists(tar):
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    shutil.copy(tar, f'{tmp}/c.tar.gz')
    with tarfile.open(f'{tmp}/c.tar.gz') as t: t.extractall(tmp, filter='data')
    src = next((root for root,_,fs in os.walk(tmp) if 'chroma.sqlite3' in fs), None)
    print('▶ tar 내 sqlite:', src)
    if src: list_collections(src)
elif os.path.isdir(f'{DRIVE}/chroma_db'):
    print('▶ 드라이브 폴더 chroma'); list_collections(f'{DRIVE}/chroma_db')
print('\n※ count > 0 인 이름을 [1] config 의 COLLECTION_NAME 에 반영하세요.')

Mounted at /content/drive
▶ tar 내 sqlite: /content/_chroma_probe/chroma_db
  경로: /content/_chroma_probe/chroma_db
  - bidmate_chunks_all               count=10,068
  - bidmate_retrieval_v1             count=26,316

※ count > 0 인 이름을 [1] config 의 COLLECTION_NAME 에 반영하세요.


In [4]:
# [1] 마운트 + chroma 로컬 폴더 준비 (★ 빈 컬렉션 재사용 방지 — 0점 버그 수정 핵심)
#  ┌─────────────────────────────────────────────────────────────────┐
#  │ 청킹 전환 스위치 — 아래 블록만 토글하면 chunks_all ↔ retrieval_v1.  │
#  └─────────────────────────────────────────────────────────────────┘
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, tarfile, gc, chromadb, json
DRIVE = '/content/drive/MyDrive/data/bidmate'

# ===== 청킹 선택 (chunks_all 기본) ==================================
CHUNK_TAG       = 'chunks_all'
CHUNK_FILE      = 'chunks/chunks_all.json'
BM25_FILE       = 'bm25/bm25_index_bidmate_chunks_all_A-2.pkl'
CHROMA_TAR      = 'chroma_db.tar.gz'
CHROMA_SUBDIR   = 'chroma_db'
COLLECTION_NAME = 'bidmate_chunks_all'      # tar 안 정상 count ≈ 10,068
EXPECT_MIN      = 8000                       # ★ 이 값 이상이어야 "정상 적재"로 인정 (빈 컬렉션 0 차단)
# # ----- retrieval_v1(≈26,316) 로 돌릴 때 -----
# COLLECTION_NAME = 'bidmate_retrieval_v1'
# EXPECT_MIN      = 20000
# =====================================================================

LOCAL      = f'/content/bidmate_{CHUNK_TAG}'
CHROMA_DIR = f'{LOCAL}/{CHROMA_SUBDIR}'
os.makedirs(LOCAL, exist_ok=True)
COLLECTION_NAME = COLLECTION_NAME.strip()

def _clear():
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass

def _count(path, name):
    _clear()
    try: return chromadb.PersistentClient(path=path).get_collection(name).count()
    except Exception: return -1

def _ensure_chroma():
    # ★ 수정: count 가 EXPECT_MIN 이상일 때만 재사용. 0 이나 미달이면 무조건 tar 재해제.
    n = _count(CHROMA_DIR, COLLECTION_NAME)
    if n >= EXPECT_MIN:
        print(f'chroma 재사용: {CHROMA_DIR} (count={n:,})'); return
    print(f'재적재 필요 (현재 count={n}) — 기존 폴더 비우고 tar 재해제')
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)

    tar = f'{DRIVE}/{CHROMA_TAR}'
    assert os.path.exists(tar), f'tar 없음: {tar}'
    sz = os.path.getsize(tar)/1e6
    assert sz > 1, f'❌  tar 이 너무 작음({sz:.2f}MB) — 빈 파일 의심: {tar}'
    print(f'tar 해제 중: {CHROMA_TAR} ({sz:.0f}MB)')
    with tarfile.open(tar) as t:
        names = t.getnames()
        assert any('chroma.sqlite3' in nm for nm in names), '❌  tar 안에 sqlite 없음'
        t.extractall(LOCAL, filter='data')
    # 풀린 sqlite 위치 → CHROMA_DIR 로 정렬
    src = next((root for root,_,fs in os.walk(LOCAL) if 'chroma.sqlite3' in fs), None)
    assert src, '해제 후 sqlite 못 찾음'
    if os.path.abspath(src) != os.path.abspath(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR, ignore_errors=True); shutil.move(src, CHROMA_DIR)
    print('chroma 준비 완료:', CHROMA_DIR)

    # ★ 해제 직후 재검증: 여전히 비어 있으면 즉시 중단 (조용히 0점으로 넘어가지 않게)
    n2 = _count(CHROMA_DIR, COLLECTION_NAME)
    assert n2 >= EXPECT_MIN, (
        f'❌  tar 해제 후에도 컬렉션 {COLLECTION_NAME} count={n2} (< {EXPECT_MIN}). '
        f'[0b] 진단에서 tar 안 컬렉션 이름/개수를 다시 확인하세요.')
    print(f'재검증 통과: {COLLECTION_NAME} count={n2:,}')

_ensure_chroma()

_cl = chromadb.PersistentClient(path=CHROMA_DIR)
_names = [c.name for c in _cl.list_collections()]
assert COLLECTION_NAME in _names, f'컬렉션 {COLLECTION_NAME} 없음. 존재: {_names}'
EXPECT_N = _cl.get_collection(COLLECTION_NAME).count()
assert EXPECT_N >= EXPECT_MIN, f'❌  EXPECT_N={EXPECT_N} 가 비정상 — 재실행 필요'
print(f'[{CHUNK_TAG}] 컬렉션={COLLECTION_NAME} | count(기준)={EXPECT_N:,} | 그 외={_names}')

# 청크/bm25/eval 로컬 복사
for rel in [CHUNK_FILE, BM25_FILE, 'eval/eval_retrieval_579.csv']:
    s=f'{DRIVE}/{rel}'; d=f'{LOCAL}/{rel}'
    assert os.path.exists(s), f'원본 없음: {s}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d): shutil.copy(s, d)

# 후속 셀 고정 경로 동기화 + 출력 폴더를 청킹별로 분리
FIXED='/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
shutil.copy(f'{LOCAL}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
OUT_TAG_DIR = f'{FIXED}/outputs/{CHUNK_TAG}'
os.makedirs(OUT_TAG_DIR, exist_ok=True)
print('eval 동기화 / 출력폴더:', OUT_TAG_DIR)
print('로컬:', os.listdir(LOCAL))

_CHUNK_TAG,_CHUNK_FILE,_BM25_FILE,_COLLECTION_NAME,_EXPECT_N,_CHROMA_DIR,_OUT_TAG_DIR = \
    CHUNK_TAG,CHUNK_FILE,BM25_FILE,COLLECTION_NAME,EXPECT_N,CHROMA_DIR,OUT_TAG_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
재적재 필요 (현재 count=-1) — 기존 폴더 비우고 tar 재해제
tar 해제 중: chroma_db.tar.gz (311MB)
chroma 준비 완료: /content/bidmate_chunks_all/chroma_db
재검증 통과: bidmate_chunks_all count=10,068
[chunks_all] 컬렉션=bidmate_chunks_all | count(기준)=10,068 | 그 외=['bidmate_chunks_all', 'bidmate_retrieval_v1']
eval 동기화 / 출력폴더: /content/bidmate/outputs/chunks_all
로컬: ['chunks', 'chroma_db', 'bm25', 'eval']


In [5]:
# [2] config 주입 — [1] 에서 정한 값 사용
import sys, types, os
from pathlib import Path
CODE='/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME']='/content/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

cfg=types.ModuleType('config')
cfg.ENV='colab'
cfg.PROJECT_ROOT=Path(f'/content/bidmate_{_CHUNK_TAG}')
cfg.DATASET_DIR=cfg.PROJECT_ROOT
cfg.CHUNKS_PATH=cfg.PROJECT_ROOT/_CHUNK_FILE
cfg.CHROMA_PATH=Path(_CHROMA_DIR)
cfg.BM25_PATH=cfg.PROJECT_ROOT/_BM25_FILE
cfg.EVAL_PATH=cfg.PROJECT_ROOT/'eval'
cfg.RESULT_DIR=cfg.PROJECT_ROOT/'eval_results'
cfg.ADAPTER_PATH=Path('/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter')
cfg.LOG_PATH=cfg.PROJECT_ROOT/'web_user_access.log'
cfg.BASE_MODEL_ID='microsoft/Phi-4-mini-instruct'
cfg.LLM_MODEL='microsoft/Phi-4-mini-instruct'
cfg.EMBED_MODEL_ID='nlpai-lab/KURE-v1'
cfg.RERANKER_ID='BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE=300; cfg.MAX_TOKENS_GENERATE=800
cfg.COLLECTION_NAME=_COLLECTION_NAME
cfg.EXPECT_N=_EXPECT_N
cfg.OUT_TAG_DIR=_OUT_TAG_DIR
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64
sys.modules['config']=cfg
assert cfg.EXPECT_N > 0, '❌  EXPECT_N=0 — [1] 을 다시 실행해 chroma 를 채우세요'
print(f'config OK → tag={_CHUNK_TAG} | col={cfg.COLLECTION_NAME} | expect={cfg.EXPECT_N:,}')
print('  CHROMA:', cfg.CHROMA_PATH); print('  BM25  :', cfg.BM25_PATH); print('  OUT   :', cfg.OUT_TAG_DIR)

config OK → tag=chunks_all | col=bidmate_chunks_all | expect=10,068
  CHROMA: /content/bidmate_chunks_all/chroma_db
  BM25  : /content/bidmate_chunks_all/bm25/bm25_index_bidmate_chunks_all_A-2.pkl
  OUT   : /content/bidmate/outputs/chunks_all


In [6]:
# [3] pre-check — chroma count > 0 을 최우선 검증 (★ 0점 버그 수정: 0 통과 차단)
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  GPU :', torch.cuda.get_device_name(0))
    print('  VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for k,p in {'CHUNKS':C.CHUNKS_PATH,'CHROMA':C.CHROMA_PATH,'BM25':C.BM25_PATH,
            'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'{"OK" if Path(p).exists() else "MISSING":8}{k:8}{p}')

with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    n_chunks = len(json.load(f))

gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
n_chroma = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME).count()

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
n_bm25 = len(bm['chunk_ids'])
print(f'\n청크JSON {n_chunks:,} | chroma {n_chroma:,} | bm25 {n_bm25:,}  (기준 chroma={C.EXPECT_N:,})')
if Path(C.ADAPTER_PATH).exists():
    print('어댑터:', os.listdir(C.ADAPTER_PATH)[:6])

# ★ 최우선: chroma 가 비어 있으면 즉시 중단 (이전 0점의 근본 원인)
assert n_chroma > 0, '❌  chroma 비어있음 (count=0) — [1] 을 재실행해 tar 를 재해제하세요'
assert n_chroma == C.EXPECT_N, f'chroma count 불일치: {n_chroma:,} != {C.EXPECT_N:,}'
if n_bm25 != n_chroma:
    print(f'⚠️  bm25({n_bm25:,}) != chroma({n_chroma:,}) — 하이브리드 사용 시 인덱스 정합 확인')
if n_chunks != n_chroma:
    print(f'ℹ️  청크 JSON({n_chunks:,}) != chroma({n_chroma:,}) — 적재 시 분할/중복(정상일 수 있음)')
print('✅  pre-check 통과 (chroma 비어있지 않음)')

CUDA: True
  GPU : Tesla T4
  VRAM: 15.6 GB
OK      CHUNKS  /content/bidmate_chunks_all/chunks/chunks_all.json
OK      CHROMA  /content/bidmate_chunks_all/chroma_db
OK      BM25    /content/bidmate_chunks_all/bm25/bm25_index_bidmate_chunks_all_A-2.pkl
OK      EVAL    /content/bidmate_chunks_all/eval/eval_retrieval_579.csv
OK      ADAPTER /content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter

청크JSON 10,127 | chroma 10,068 | bm25 10,068  (기준 chroma=10,068)
어댑터: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
ℹ️  청크 JSON(10,127) != chroma(10,068) — 적재 시 분할/중복(정상일 수 있음)
✅  pre-check 통과 (chroma 비어있지 않음)


In [7]:
# [4] 서빙 모듈 로드 + retriever 조립
import importlib.util, sys, pickle, gc, os
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE = '/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path = f'{CODE}/{name}.py'
    assert os.path.exists(path), f'파일 없음: {path}'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

Rtv = load_module('retrieval')
print('retrieval 로드 OK')
Rtv.DEVICE = DEVICE
all_chunks = Rtv.load_chunks()
Rtv.ALL_AGENCIES = list({c['metadata'].get('agency','') for c in all_chunks
                         if c['metadata'].get('agency','')})
print(f'load_chunks: {len(all_chunks):,} | agencies: {len(Rtv.ALL_AGENCIES)}')

embed_model = SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE,
                                  cache_folder='/content/hf_cache/hub')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
collection = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
_cnt = collection.count(); print('chroma count:', f'{_cnt:,}')
# ★ 수정: 0 이면 즉시 중단 (이전엔 EXPECT_N=0 이라 0==0 통과)
assert _cnt > 0, f'❌  chroma count=0 — [1] 재실행 필요'
assert _cnt == C.EXPECT_N, f'chroma count={_cnt:,} (기대 {C.EXPECT_N:,})'

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
reranker = CrossEncoder(C.RERANKER_ID, device=DEVICE)
retriever = Rtv.BidMateRetriever(
    collection=collection, bm25_index=bm['index'],
    bm25_chunk_ids=bm['chunk_ids'], bm25_texts=bm['texts'],
    embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
)
Rtv.retriever = retriever
print('✅  retriever 초기화 완료')

retrieval 로드 OK
load_chunks: 10,127 | agencies: 406


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅  retriever 초기화 완료


In [8]:
# [4b] generator 로드 (서빙 generation.py + FT Phi 어댑터)
import torch
Gen = load_module('generation')
print('generation 로드 OK')
generator = Gen.init_generator(Rtv.get_context)
Gen.generator = generator
if torch.cuda.is_available():
    print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
          '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅  generator 초기화 완료 (Phi-4-mini + LoRA)')

generation 로드 OK


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

VRAM 사용: 12.3 / 15.6 GB
✅  generator 초기화 완료 (Phi-4-mini + LoRA)


In [9]:
from peft import PeftModel

# 노트북에서 generator를 초기화하는 코드 아래에 배치하세요.
if hasattr(generator.client.messages, "_model") and isinstance(generator.client.messages._model, PeftModel):
    print("LoRA 어댑터를 끄고 원본 베이스 모델로 복구합니다.")
    generator.client.messages._model = generator.client.messages._model.unload()

LoRA 어댑터를 끄고 원본 베이스 모델로 복구합니다.


In [10]:
# [4c-diag] 검색 0건 원인 분리 — 메타필터 vs chroma
r = eval_df.iloc[0]
hist = _ph(r.get('history','')); mf = _pm(r.get('metadata_filter',''))
rewritten = generator._rewrite_query(r['question'], hist or None)

print('=== 진단 ===')
print('metadata_filter(raw):', repr(r.get('metadata_filter','')))
print('parsed mf           :', mf)

# 1) 메타필터 OFF 로 검색 — 여기서 결과 나오면 범인은 메타필터
rr_nofilter = retriever.retrieve(rewritten, meta_filter=None)
print('필터 OFF top_chunks :', len(rr_nofilter.get('top_chunks', [])))

# 2) 메타필터 ON
rr_filter = retriever.retrieve(rewritten, meta_filter=mf)
print('필터 ON  top_chunks :', len(rr_filter.get('top_chunks', [])))

# 3) mf 의 키/값이 chroma 메타데이터에 실제 존재하는지
if mf:
    sample = collection.get(limit=3, include=['metadatas'])
    print('chroma 메타 키 예시:', list(sample['metadatas'][0].keys()) if sample['metadatas'] else '없음')
    print('mf 가 요구하는 키  :', list(mf.keys()) if isinstance(mf, dict) else mf)

NameError: name 'eval_df' is not defined

In [11]:
# [4c-diag2] retrieve 우회 — chroma/임베딩 단독 점검
import numpy as np

# 1) chroma 가 자체 임베딩 없이 query_texts 로 검색되는지 (적재 시 임베딩 함수 유무 확인)
try:
    q_txt = collection.query(query_texts=[rewritten], n_results=5)
    print('query_texts 결과 ids:', len(q_txt['ids'][0]))
except Exception as e:
    print('query_texts 실패:', repr(e))

# 2) KURE-v1 로 직접 임베딩 → query_embeddings 로 검색
emb = embed_model.encode([rewritten], normalize_embeddings=True)
print('쿼리 임베딩 차원:', emb.shape)

# chroma 에 적재된 벡터 차원
peek = collection.get(limit=1, include=['embeddings'])
loaded_dim = len(peek['embeddings'][0]) if peek['embeddings'] else None
print('chroma 적재 차원:', loaded_dim)

q_emb = collection.query(query_embeddings=emb.tolist(), n_results=5)
print('query_embeddings 결과 ids:', len(q_emb['ids'][0]))
if q_emb['ids'][0]:
    print('  거리 샘플:', q_emb['distances'][0][:3])
    print('  메타 agency 샘플:', [m.get('agency') for m in q_emb['metadatas'][0][:3]])

query_texts 실패: NameError("name 'rewritten' is not defined")


NameError: name 'rewritten' is not defined

In [12]:
# [4c-diag3] query_embeddings 단독 — numpy 진리값 버그 제거
emb = embed_model.encode([rewritten], normalize_embeddings=True)
print('쿼리 임베딩 차원:', emb.shape)

q_emb = collection.query(query_embeddings=emb.tolist(), n_results=5)
print('query_embeddings 결과 ids:', len(q_emb['ids'][0]))
if len(q_emb['ids'][0]) > 0:
    print('  거리 샘플:', q_emb['distances'][0][:3])
    print('  agency 샘플:', [m.get('agency') for m in q_emb['metadatas'][0][:3]])
    print('  original_name 샘플:', [m.get('original_name','(키없음)') for m in q_emb['metadatas'][0][:3]])

NameError: name 'rewritten' is not defined

In [13]:
# [4-patch] _build_chroma_where 키매핑 제거 (chunks_all 은 'agency' 키 사용)
def _bcw(self, meta_filter):
    if not meta_filter: return None
    conds = []
    for key, val in meta_filter.items():
        if not val: continue
        if isinstance(val, dict):   conds.append({key: val})
        elif isinstance(val, list): conds.append({key: {"$in": [str(v) for v in val]}})
        else:                       conds.append({key: {"$eq": str(val)}})
    if not conds: return None
    return conds[0] if len(conds)==1 else {"$and": conds}
import types
retriever._build_chroma_where = types.MethodType(_bcw, retriever)
print('patched. test:', retriever._build_chroma_where({'agency':'한국가스공사'}))

patched. test: {'agency': {'$eq': '한국가스공사'}}


In [14]:
rr = retriever.retrieve(rewritten, meta_filter={'agency':'한국가스공사'})
print('필터 ON top_chunks:', len(rr['top_chunks']))
print('agency 확인:', [c['metadata'].get('agency') for c in rr['top_chunks']])

NameError: name 'rewritten' is not defined

In [15]:
# [4c-diag4] 패치 적용 여부 + retrieve 단계별 0건 추적
# 1) 패치 확인
print('where 출력:', retriever._build_chroma_where({'agency':'한국가스공사'}))

# 2) retrieve verbose 분해 — 각 단계 카운트
mf = {'agency':'한국가스공사'}
where = retriever._build_chroma_where(mf)
allowed = retriever._filter_bm25_ids(mf)
print('where        :', where)
print('bm25 allowed :', None if allowed is None else len(allowed))

dense = retriever._dense_search(rewritten, where)
print('dense 결과   :', len(dense))

sparse = retriever._sparse_search(rewritten, allowed)
print('sparse 결과  :', len(sparse))

# 3) D타입 컷오프 의심 — 0.25 미만 컷 때문인지 확인
rr = retriever.retrieve(rewritten, meta_filter=mf)
print('최종 top_chunks:', len(rr['top_chunks']))
print('dense_ids in result:', len(rr['dense_ids']), '| sparse_ids:', len(rr['sparse_ids']))

where 출력: {'agency': {'$eq': '한국가스공사'}}
where        : {'agency': {'$eq': '한국가스공사'}}
bm25 allowed : 9


NameError: name 'rewritten' is not defined

In [16]:
got = collection.get(where={'agency': {'$eq':'한국가스공사'}}, limit=3)
print('정확매칭 건수:', len(got['ids']))
# 0이면 부분일치로 후보 확인
import collections as _c
allm = collection.get(include=['metadatas'])['metadatas']
ags = [m.get('agency','') for m in allm if '가스' in m.get('agency','')]
print('가스 포함 agency 후보:', _c.Counter(ags).most_common(5))

정확매칭 건수: 3
가스 포함 agency 후보: [('한국가스공사', 9), ('한국가스안전공사', 1)]


In [17]:
# [4-patch2] D타입 컷오프를 sigmoid 스케일로 교체 (A 방식, 코랩 인메모리 패치)
import types, math

def _retrieve_sigcut(self, query, meta_filter=None, verbose=False):
    if meta_filter is None:
        meta_filter = __import__('retrieval').parse_metadata_filter(query)
    where           = self._build_chroma_where(meta_filter)
    allowed_indices = self._filter_bm25_ids(meta_filter)
    sub_queries     = self._decompose_query(query)
    if len(sub_queries) > 1:
        dense_ids, sparse_ids = self._multi_retrieve(sub_queries, where, allowed_indices, original_query=query)
    else:
        dense_ids  = self._dense_search(query, where)
        sparse_ids = self._sparse_search(query, allowed_indices)
    ranked  = self._rrf_fusion(dense_ids, sparse_ids)
    boosted = self._soft_boost(ranked)
    boosted = self._mmr_rerank(boosted, query=query)
    boosted = self._rerank(boosted, query=query)

    if len(sub_queries) > 1:
        per_agency = max(2, 5 // len(sub_queries))   # TOP_K=5
        agency_counts, top5 = {}, []
        for cid, score in boosted:
            meta = self.chunk_meta_map.get(cid, {})
            ag = meta.get("agency", meta.get("organization_cleaned", ""))
            if agency_counts.get(ag, 0) < per_agency:
                top5.append((cid, score)); agency_counts[ag] = agency_counts.get(ag, 0) + 1
            if len(top5) >= 5: break
    else:
        top5 = boosted[:5]

    # ★ A 방식: reranker logit → sigmoid 후 0.5 미만이면 근거없음(D타입 거절)
    def _sig(x): return 1/(1+math.exp(-x))
    SIG_TH = 0.5
    if top5 and _sig(top5[0][1]) < SIG_TH:
        top5 = []

    return {
        "context"    : self._build_context(top5),
        "top_chunks" : [{"rank": i+1, "chunk_id": cid, "boosted_score": sc,
                         "text": self.chunk_text_map.get(cid,""), "metadata": self.chunk_meta_map.get(cid,{})}
                        for i, (cid, sc) in enumerate(top5)],
        "meta_filter": meta_filter,
        "dense_ids"  : dense_ids,
        "sparse_ids" : sparse_ids,
        "sub_queries": sub_queries,
    }

retriever.retrieve = types.MethodType(_retrieve_sigcut, retriever)

# 검증
rr = retriever.retrieve(rewritten, meta_filter={'agency':'한국가스공사'})
print('top_chunks:', len(rr['top_chunks']))
print('상위 raw score:', [round(s,4) for _,s in [(c['chunk_id'],c['boosted_score']) for c in rr['top_chunks']]][:3])
print('agency:', [c['metadata'].get('agency') for c in rr['top_chunks']][:3])

NameError: name 'rewritten' is not defined

In [ ]:
r = eval_df.iloc[0]
print('GT docs   :', r['ground_truth_docs'])
print('source_file:', [c['metadata'].get('source_file') for c in rr['top_chunks'][:3]])

In [ ]:
# [4-patch3] _get_hwp_context 의 original_name → source_file (C타입 HWP 보강용)
import retrieval as _R
_orig_get_hwp = _R._get_hwp_context
def _get_hwp_src(query, top_chunks, embed_model, top_docs=2):
    for c in top_chunks:
        m = c.get("metadata", {})
        if "original_name" not in m and m.get("source_file"):
            m["original_name"] = m["source_file"]   # alias 주입
    return _orig_get_hwp(query, top_chunks, embed_model, top_docs)
_R._get_hwp_context = _get_hwp_src
print('hwp context source_file alias 적용')

In [18]:
print(eval_df['type'].value_counts().to_dict())

NameError: name 'eval_df' is not defined

In [19]:
# [4c] 스모크 테스트 — ★ 검색이 실제로 문서를 가져오는지부터 확인 (0점 조기 감지)
import time, pandas as pd, json, ast
eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

r = eval_df.iloc[0]
hist = _ph(r.get('history','')); mf = _pm(r.get('metadata_filter',''))

# 1) 검색 단독 확인 — top_chunks 가 비면 chroma/임베딩 쪽 문제
rewritten = generator._rewrite_query(r['question'], hist or None)
rr = retriever.retrieve(rewritten, meta_filter=mf)
top = rr.get('top_chunks', [])
print(f'[검색] rewritten={rewritten[:50]!r}')
print(f'[검색] top_chunks 개수: {len(top)}')
assert len(top) > 0, '❌  검색 결과 0건 — chroma 가 비었거나 메타필터가 과도. [1]/[3] 재확인'
names = [c['metadata'].get('source_file','') for c in top]
print(f'[검색] retrieved_names: {names}')

# 2) 생성 확인
t=time.time()
out = generator.generate(r['question'], history=hist or None, meta_filter=mf)
dt=time.time()-t
print(f'\n[{r["type"]}] {r["question"][:40]}')
print('답변:', out['answer'][:200])
print(f'\n1건 소요: {dt:.1f}초 → 579행 예상 {dt*579/3600:.1f}시간')

# 3) 정답문서명 ↔ 검색결과명 포맷 비교 (매칭 가능성 사전 점검)
print('\n[매칭 점검] ground_truth_docs:', r['ground_truth_docs'])
print('[매칭 점검] retrieved_names   :', json.dumps(names, ensure_ascii=False))

[검색] rewritten="한국가스공사 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 예산 규모입니까?"
[검색] top_chunks 개수: 5
[검색] retrieved_names: ['한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp']


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[A] 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 
답변: 제공된 문서에서 확인할 수 없습니다.

[출처]
  [1] 한국가스공사 2024 (score: 0.1365)
  [2] 한국가스공사 2024 (score: 0.1252)
  [3] 한국가스공사 2024 (score: 0.0797)
  [4] 한국가스공사 2024 (score: 0.0439)
  [5] 한국가스공사 2024 (score: 0.0024)

1건 소요: 5.8초 → 579행 예상 0.9시간

[매칭 점검] ground_truth_docs: ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]
[매칭 점검] retrieved_names   : ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"]


In [20]:
# [5] 579행 생성 — ★ question 기준으로 done 판정 (id 는 중복/부적합)
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm
import config as _C

OUT_DRIVE=_C.OUT_TAG_DIR; OUT_LOCAL=_C.OUT_TAG_DIR
os.makedirs(OUT_DRIVE,exist_ok=True); os.makedirs(OUT_LOCAL,exist_ok=True)
GEN_PATH=f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv'

eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), '| 고유 question:', eval_df['question'].nunique(),
      '| 타입:', eval_df['type'].value_counts().to_dict())

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

# ★ 체크포인트 재사용도 question 기준
done, records = set(), []
if os.path.exists(GEN_PATH):
    prev = pd.read_csv(GEN_PATH)
    prev = prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)]
    prev = prev.drop_duplicates(subset='question')
    records = prev.to_dict('records'); done = set(prev['question'])
    print('체크포인트 재사용:', len(done))

pending = eval_df.drop_duplicates(subset='question')
pending = pending[~pending['question'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time()
    rewritten = generator._rewrite_query(q, hist or None)
    rr = retriever.retrieve(rewritten, meta_filter=mf)
    retr_ms = round((time.time()-t0)*1000)
    top = rr['top_chunks']
    t1=time.time()
    out = generator.generate(q, history=hist or None, meta_filter=mf)
    gen_ms = round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],
        'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('source_file','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH,index=False,encoding='utf-8-sig')

gen_df=pd.DataFrame(records).drop_duplicates(subset='question')
gen_df.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
gen_df.to_csv(f'{OUT_DRIVE}/e2e_kure_phi_ft_579.csv',index=False,encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df), '| 고유 question:', gen_df['question'].nunique())

평가셋: 579 | 고유 question: 578 | 타입: {'B': 214, 'A': 172, 'D': 65, 'E': 65, 'C': 63}
신규: 578


생성:   0%|          | 0/578 [00:00<?, ?it/s]

✅ 생성 완료: 578 | 고유 question: 578


In [21]:
# [5b] 생성 결과 무결성 점검 (question 기준)
import pandas as pd
import config as _C
GEN_PATH = f'{_C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
df = pd.read_csv(GEN_PATH)
ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

print('저장된 행:', len(df), '| 고유 question:', df['question'].nunique())
empty_ans = df['answer'].isna().sum() + (df['answer'].astype(str).str.len()==0).sum()
print('answer 빈 행:', empty_ans)

done_q = set(df['question']); ev_q = set(ev['question'])
print('eval 에 있는데 저장 안 된 question:', len(ev_q - done_q))
print('중복 question 수:', df['question'].duplicated().sum())

# ★ retrieved_names 가 전부 비었는지 (= 검색 실패 = 0점 전조) 확인
import json
def _empty_names(raw):
    try: v=json.loads(raw)
    except Exception: return True
    return (not isinstance(v,list)) or len(v)==0 or all(not str(x).strip() for x in v)
n_empty = df['retrieved_names'].apply(_empty_names).sum()
print(f'retrieved_names 비어있는 행: {n_empty} / {len(df)}')
assert n_empty < len(df)*0.5, '❌  검색결과가 절반 이상 비어있음 — chroma/메타필터 재점검 필요'
print('✅ 무결성 점검 통과')

저장된 행: 578 | 고유 question: 578
answer 빈 행: 0
eval 에 있는데 저장 안 된 question: 0
중복 question 수: 0
retrieved_names 비어있는 행: 1 / 578
✅ 무결성 점검 통과


In [22]:
# [6] Retrieval 지표 — Hit@5/MRR/nDCG (ground_truth_docs vs retrieved_names)
import pandas as pd, json, ast, math, os
import config as _C
OUT_LOCAL=_C.OUT_TAG_DIR; OUT_DRIVE=_C.OUT_TAG_DIR
gen_df = pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv')

def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()

def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    hit=1.0 if rank else 0.0; mrr=1.0/rank if rank else 0.0
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':hit,'mrr':mrr,'ndcg':dcg/idcg if idcg else 0.0})

rm = gen_df.join(gen_df.apply(rmetrics, axis=1))
valid = rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행 (정답문서 있는 행)')
print('전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))

# ★ 전부 0 이면 자동 진단: 정규화 후 매칭이 실제로 일어나는지 샘플 출력
if valid['hit@5'].mean() == 0.0:
    print('\n⚠️  Hit@5 전체 0 — 매칭 진단 샘플 (정규화 후 비교):')
    for _, row in valid.head(3).iterrows():
        gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
        got=[_norm(x) for x in _tolist(row['retrieved_names'])][:5]
        print('  GT :', gts)
        print('  GOT:', got)
        print('  교집합:', set(gts) & set(got), '\n')
    print('  → GT/GOT 형태가 다르면 _norm() 규칙(확장자/공백/대소문자 외 구분자)을 조정하세요.')

s = valid.groupby('type')[['hit@5','mrr','ndcg']].mean()
s.loc['ALL'] = valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT_DRIVE}/retrieval_metrics_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

대상 573행 (정답문서 있는 행)
전체: {'hit@5': 0.9005, 'mrr': 0.8354, 'ndcg': 1.5332}

타입별:
        hit@5     mrr    ndcg
type                        
A     0.9240  0.8733  1.9774
B     0.9481  0.8664  1.0503
C     0.7419  0.6815  1.5773
D     0.8906  0.8411  1.8928
E     0.8438  0.7753  1.5440
✅ 저장


In [ ]:
# [7] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
import shutil, tarfile, gc, chromadb
from tqdm.auto import tqdm
import config as _C

# Colab 환경 체크 및 드라이브 마운트 오류 수정
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # 로컬 환경인 경우 무시

from openai import AsyncOpenAI

nest_asyncio.apply()
OUT_LOCAL = _C.OUT_TAG_DIR
OUT_DRIVE = _C.OUT_TAG_DIR

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY 필요'

_M = 'gpt-5.4-mini'
_client = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM = asyncio.Semaphore(15)
_RETRY = 3

_JP = {
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n"
                "[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n"
                "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n"
             "[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n"
             "[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n"
             "[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n"
             "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n"
               "[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n"
               "[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n"
                     "[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n"
                     "[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n"
                  "[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n"
                  "[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}

def _parse(raw):
    if not raw: return None
    m = re.search(r'점수\s*:\s*(\d)', raw)
    if m: return int(m.group(1))
    s = raw.strip()
    if s.isdigit() and 1 <= int(s) <= 5: return int(s)
    d = re.findall(r'\b[1-5]\b', raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r = await _client.chat.completions.create(
                    model=_M,
                    messages=[{'role': 'user', 'content': prompt}],
                    max_completion_tokens=20,
                    timeout=15
                )
                return _parse(r.choices[0].message.content)
            except Exception:
                if a == _RETRY - 1: return None
                await asyncio.sleep(2**a)

async def score_one(q, ctx, ans, gt=None):
    tasks, none_keys = {}, []
    for m in ('faithfulness', 'relevance', 'rejection'):
        tasks[m] = _ask(_JP[m].format(context=ctx, query=q, answer=ans))
    for m in ('correctness', 'context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m] = _ask(_JP[m].format(ground_truth=gt, answer=ans, context=ctx))
        else:
            none_keys.append(m)
    tasks['context_precision'] = _ask(_JP['context_precision'].format(query=q, context=ctx))

    vals = await asyncio.gather(*tasks.values())
    res = dict(zip(tasks.keys(), vals))
    for k in none_keys: res[k] = None
    return res

_MET = ['faithfulness', 'relevance', 'rejection', 'correctness', 'context_precision', 'context_recall']
JUDGE_PATH = f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv'

async def process_row(row):
    ans = row['answer']
    base = {'id': row['id'], 'question': row['question'], 'type': row['type'], 'difficulty': row['difficulty']}
    if not isinstance(ans, str) or '오류' in str(ans)[:30]:
        for m in _MET: base[m] = None
    else:
        scores = await score_one(row['question'], row['retrieved_context'], ans, row.get('ground_truth_answer'))
        base.update(scores)
    return base

async def run_judge():
    gdf = pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
    done, rows = set(), []

    if os.path.exists(JUDGE_PATH):
        ck = pd.read_csv(JUDGE_PATH)
        ck = ck[ck['relevance'].notna()].drop_duplicates(subset='question')
        rows = ck.to_dict('records')
        done = set(ck['question'])
        print('judge 체크포인트 완료 문항 수:', len(done))

    pending = gdf[~gdf['question'].isin(done)]
    print('judge 평가 대상 신규 문항 수:', len(pending))

    # 50개 단위 청크 분할 병렬 처리로 속도 개선
    chunk_size = 50
    pending_records = pending.to_dict('records')

    for i in range(0, len(pending_records), chunk_size):
        chunk = pending_records[i:i+chunk_size]
        tasks = [process_row(row) for row in chunk]

        # 설정한 세마포어(15) 한도 내에서 동시 처리됨
        chunk_results = await asyncio.gather(*tasks)
        rows.extend(chunk_results)

        # 50개 단위 주기적 저장
        pd.DataFrame(rows).to_csv(JUDGE_PATH, index=False, encoding='utf-8-sig')
        print(f"진행 완료: {min(i + chunk_size, len(pending_records))} / {len(pending_records)}")

    out = pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH, index=False, encoding='utf-8-sig')
    out.to_csv(f'{OUT_DRIVE}/quant_scores_kure_phi_ft.csv', index=False, encoding='utf-8-sig')
    print('✅ judge 최종 완료:', len(out))
    return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
judge 평가 대상 신규 문항 수: 578
진행 완료: 50 / 578
진행 완료: 100 / 578
진행 완료: 150 / 578
진행 완료: 200 / 578
진행 완료: 250 / 578
진행 완료: 300 / 578


In [25]:
# [8] Generation 요약
import pandas as pd
import config as _C
OUT_LOCAL=_C.OUT_TAG_DIR; OUT_DRIVE=_C.OUT_TAG_DIR
judge_df=pd.read_csv(f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean()
s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT_DRIVE}/generation_summary_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

전체: {'faithfulness': nan, 'relevance': nan, 'rejection': nan, 'correctness': nan, 'context_precision': nan, 'context_recall': nan}

타입별:
       faithfulness  relevance  rejection  correctness  context_precision  \
type                                                                       
A              NaN        NaN        NaN          NaN                NaN   
B              NaN        NaN        NaN          NaN                NaN   
C              NaN        NaN        NaN          NaN                NaN   
D              NaN        NaN        NaN          NaN                NaN   
E              NaN        NaN        NaN          NaN                NaN   

      context_recall  
type                  
A                NaN  
B                NaN  
C                NaN  
D                NaN  
E                NaN  
✅ 저장


In [26]:
# [9] Release Gate
import pandas as pd
import config as _C
OUT_DRIVE=_C.OUT_TAG_DIR
retr=pd.read_csv(f'{OUT_DRIVE}/retrieval_metrics_kure_phi_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT_DRIVE}/generation_summary_kure_phi_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')

print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    if m in genm.columns:
        print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

RETRIEVAL (전체)
  Hit@5 0.901 → PASS
  MRR   0.835 → PASS
  nDCG  1.533 → GOOD
타입별 MRR
  A 0.873 → FAIL
  B 0.866 → GOOD
  C 0.681 → FAIL
  D 0.841 → PASS
  E 0.775 → FAIL
GENERATION (≥3.5 PASS / ≥4.0 GOOD)
  faithfulness       nan → FAIL
  relevance          nan → FAIL
  rejection          nan → FAIL
  context_precision  nan → FAIL


In [27]:
# [10] 산출물 목록 — outputs/<tag>/ 만 안전하게 나열
import os
import config as _C
OUT_DRIVE=_C.OUT_TAG_DIR
for f in sorted(os.listdir(OUT_DRIVE)):
    p=os.path.join(OUT_DRIVE,f)
    if os.path.isfile(p):                      # ★ 폴더/.gslides 등은 건너뜀 (이전 FileNotFoundError 방지)
        print(f'{os.path.getsize(p)/1024:8.1f} KB  {f}')
    else:
        print(f'{"<dir>":>11}  {f}')

  2765.1 KB  e2e_kure_phi_ft_579.csv
     0.1 KB  generation_summary_kure_phi_ft.csv
   166.2 KB  quant_scores_kure_phi_ft.csv
     0.3 KB  retrieval_metrics_kure_phi_ft.csv


In [ ]:
# [11] 정성 분석 — 단일 시나리오(KURE+Phi FT)용
import pandas as pd, os
import config as _C
OUT_DRIVE=_C.OUT_TAG_DIR; OUT_LOCAL=_C.OUT_TAG_DIR
QUAL_DIR  = f'{OUT_DRIVE}/qual'
os.makedirs(QUAL_DIR, exist_ok=True)

gen_df   = pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv')
judge_df = pd.read_csv(f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv')
_MET = ['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
ERROR_TH = 3.0

# ★ question 기준 병합 (id 부적합)
score_cols = ['question'] + [m for m in _MET if m in judge_df.columns]
gen_df = gen_df.drop_duplicates(subset='question')
judge_df = judge_df.drop_duplicates(subset='question')
merged = gen_df.merge(judge_df[score_cols], on='question', how='left')
print(f'병합: {len(merged)}행')

# 1) 오류 역추적
mask = pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns:
        mask |= (merged[m].notna() & (merged[m] <= ERROR_TH))
mask |= merged['answer'].astype(str).str.contains('오류', na=False)
err_cols = [c for c in ['id','type','difficulty','question','ground_truth_answer',
                        'answer','retrieved_context'] + _MET if c in merged.columns]
error_df = merged[mask][err_cols].copy()
error_df.to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {len(error_df)}건 → qual_error_analysis.csv')

# 2) C타입 맥락 추적
kws = ['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask = (merged['type'] == 'C')
cmask |= merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols = [c for c in ['id','type','question','ground_truth_answer','answer',
                      'retrieved_context'] if c in merged.columns]
ctype_df = merged[cmask][c_cols].copy()
ctype_df.to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락 추적: {len(ctype_df)}건 → qual_ctype_tracking.csv')

# 3) 타입별 요약
rows = []
for t in ['A','B','C','D','E']:
    sub = merged[merged['type'] == t]
    if sub.empty: continue
    r = {'type': t, 'n': len(sub)}
    for m in _MET:
        r[m] = round(sub[m].dropna().mean(), 3) if m in sub.columns else None
    r['gen_errors'] = int(sub['answer'].astype(str).str.contains('오류', na=False).sum())
    rows.append(r)
summary_df = pd.DataFrame(rows)
summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print(f'3) 타입별 요약 → qual_summary.csv')
print('\n📊 타입별 요약')
print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')

In [ ]:
# [15] 산출물 Drive 백업 — outputs 통째로 (로컬은 런타임 종료 시 소실)
import shutil, os
import config as _C
if not os.path.ismount('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
SRC=_C.OUT_TAG_DIR
DST='/content/drive/MyDrive/data/bidmate/outputs/'+os.path.basename(SRC)
os.makedirs(os.path.dirname(DST), exist_ok=True)
if os.path.abspath(SRC)!=os.path.abspath(DST):
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
print('백업 완료:', DST)
for f in sorted(os.listdir(DST)): print('  -', f)